In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from pysheds.grid import Grid
from pysheds.sview import Raster
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import shape
from scipy.ndimage import gaussian_filter
from rasterio.features import shapes
from tqdm import tqdm

In [ ]:
watershed = gpd.read_file("GIS/WATERSHEDS.geojson")
watershed = watershed[watershed['NAME'] == 'Popes Head Creek']
watershed = watershed.to_crs('EPSG:26918')

In [ ]:
# load elevation data for particular watershed
with rasterio.open("elevation/fairfax_dem_1m.tif") as src:    
    out_image, out_transform = mask(src, watershed.geometry, crop=True)
    out_meta = src.meta.copy()
    out_meta.update({
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })

with rasterio.open("elevation/popes_head_dem.tif", 'w', **out_meta) as dest:
    dest.write(out_image)

In [ ]:
grid = Grid.from_raster("elevation/popes_head_dem.tif")
dem = grid.read_raster("elevation/popes_head_dem.tif")

In [ ]:
plt.imshow(np.where(dem == -9999, np.nan, dem), cmap='terrain')
plt.title('Popes Head Creek DEM')
plt.colorbar(label='Elevation (m)')
plt.axis('off')
plt.show()

In [ ]:
pit_filled = grid.fill_pits(dem)
flooded = grid.fill_depressions(pit_filled)
inflated = grid.resolve_flats(flooded)
flowdir = grid.flowdir(inflated)
acc = grid.accumulation(flowdir)

In [ ]:
# with rasterio.open('flowdir.tif', "w",
#     driver="GTiff",
#     height=flowdir.shape[0],
#     width=flowdir.shape[1],
#     count=1,
#     dtype=flowdir.dtype,
#     crs=src.crs,
#     transform=grid.affine) as dst:
#     dst.write(flowdir, 1)

flowdir = grid.read_raster('flowdir.tif')

In [ ]:
acc_smooth = np.log10(acc+1)
acc_smooth = gaussian_filter(acc_smooth, sigma=3)
plt.imshow(acc_smooth)
plt.axis('off')
plt.colorbar(label='log2(Accumulation + 1)')
plt.title('Popes Head Creek Smoothed Accumulation')
plt.show()

In [ ]:
plt.imshow(np.where(dem == -9999, np.nan, flowdir))
plt.axis('off')
plt.colorbar(label='Flow Direction')
plt.show()

## Subcatchment Creation Using Recorded Inlets
Every collective point of inflow should be accounted for in the model. Directly apply the recorded stormwater infrastructure data to identify points of urban inflow.

In [ ]:
infrastructure = gpd.read_file('stormnet/13.geojson')
infrastructure = infrastructure.to_crs(watershed.crs).clip(watershed)

In [ ]:
drainage_sources = ['Inlet', 'Infall']
is_source = [infrastructure['TYPE'].str.contains(source) for source in drainage_sources]
is_source = np.logical_or(*is_source)
is_source = np.where(is_source.isna(), False, is_source)
drainage = infrastructure[is_source]

In [ ]:
subcatchments = np.zeros_like(acc)
mask = np.where(dem == -9999, True, False)
area = np.sum(~mask)
subcatchment_area = 0
geometries = tqdm(drainage.geometry)
used = {0}

sink_patch = np.array([
    [  2,   4,   8  ],  # NW, N, NE
    [ 1,  -2,  16 ],  # W,  PIT,  E
    [128,  64,  32 ]   # SW, S, SE
])

for source in geometries:
    j, i = grid.affine.__invert__()*(source.x, source.y)
    i, j = round(i), round(j)
    if (l := subcatchments[i, j]):
        label = l
    else:
        label = max(used) + 1
    
    # manipulate inlet surroundings for water-intake emphasis (better delineation)
    original = flowdir[i-1:i+2, j-1:j+2].copy()
    flowdir[i-1:i+2, j-1:j+2] = sink_patch

    sub = grid.catchment(x=j, y=i, fdir=flowdir, xytype='index')
    flowdir[i-1:i+2, j-1:j+2] = original
    sub = np.where(mask, False, sub)
    mask[i, j] = True
    if not sub.sum():
        continue
    subcatchment_area += sub.sum()
    subcatchments[sub] = label
    used.add(label)
    mask = np.logical_or(mask, sub)
    geometries.set_description(f"Area covered - {subcatchment_area/area*100:.2f}%")
    

In [ ]:
polygons = []
for label in tqdm(np.unique(subcatchments)):
    if not label:
        continue
    arr = (subcatchments == label).astype(np.uint8)

    for geom, val in shapes(arr, mask=arr==1, transform=grid.affine):
        polygons.append(shape(geom))

In [ ]:
gdf = gpd.GeoDataFrame(
    {"geometry": polygons},
    crs=grid.crs
)

In [ ]:
gdf[(gdf.area > 100)].reset_index().plot(column='index', legend=True)

In [ ]:
gdf.plot()

In [ ]:
gdf.to_file("subcatchments.geojson", driver="GeoJSON")